In [1]:
from spin_lattices import KagomeLattice, SpinLattice, SquareLattice, TriangleLattice
from heisenberg_hamiltonians import HeisenbergJ1J2, SpinSystem
from pathlib import Path
import torch
from tqdm.auto import tqdm
from torch.optim import Adam
import numpy as np
import numpy.typing as npt
from misc_utils import make_unpacked_configurations

import torch
import torch.nn as nn
import torch.nn.functional as F

import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim



2023-06-20 20:12:19.816 | DEBUG    | lattice_symmetries:__init__:49 - Initializing Haskell runtime...
2023-06-20 20:12:19.819 | DEBUG    | lattice_symmetries:__init__:51 - Initializing Chapel runtime...
[Debug]   [2023-06-20 20:12:19.873 | DEBUG    | lattice_symmetries:__init__:53 - Setting Python exception handler...
LOCALE0]   Initializing chpl_kernels ...
set_python_exception_handler ...


In [25]:
def conv2d_circular(input, weight, bias=None, stride=1, padding=0, dilation=1, groups=1):
    # Apply circular padding
    if padding > 0:
        input = F.pad(input, (padding, 0, padding, 0), mode="circular")

    return F.conv2d(input, weight, bias, stride, 0, dilation, groups)


class CircularConv2d(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride=1,
        padding=0,
        dilation=1,
        groups=1,
        bias=True,
    ):
        super(CircularConv2d, self).__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels, kernel_size, stride, 0, dilation, groups, bias
        )
        self.padding = padding

    def forward(self, x):
        return F.relu(
            conv2d_circular(
                x,
                self.conv.weight,
                self.conv.bias,
                self.conv.stride,
                self.padding,
                self.conv.dilation,
                self.conv.groups,
            )
        )


class CNNRegression(nn.Module):
    def __init__(self, hidden_channels1=32, hidden_channels2=64, kernel_size=2):
        super(CNNRegression, self).__init__()
        self.conv1 = CircularConv2d(
            3, hidden_channels1, kernel_size=(kernel_size, kernel_size), padding=kernel_size
        )
        self.conv2 = CircularConv2d(
            hidden_channels1,
            hidden_channels2,
            kernel_size=(kernel_size, kernel_size),
            padding=kernel_size,
        )
        self.fc = nn.Linear(hidden_channels2, 1)  # Output a single value for regression

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.mean(dim=(2, 3))  # Average over spatial dimensions
        x = self.fc(x)
        return x

def get_X_Y(system: SpinSystem, sample=None, target='amplitude'):
    ground_state_df = system.get_df_ground_state()
    if sample:
        ground_state_df = ground_state_df.sample(sample)
    Y = torch.Tensor(np.log(np.real_if_close(ground_state_df[target].to_numpy()).astype(np.float32)))
    # Y = Y / Y.mean()
    X = torch.Tensor(
        system.lattice.spin_config_to_tensor(ground_state_df.index.to_numpy()).astype(np.float32)
    ).permute(0, 3, 1, 2)
    return X, Y

def overlap(x, y):
    return torch.sum(x * y) / torch.sqrt(torch.sum(x ** 2) * torch.sum(y ** 2))

In [26]:
J2 = 0.8
lat_source = KagomeLattice(2, 3)
system_source = HeisenbergJ1J2(
    lattice=lat_source,
    J2=J2,
    use_symmetries=True,
    spin_inversion=1,
    ground_state_cache_dir=Path("groundstates"),
    skip_symmetries_whitelist=True,
)

lat_target = KagomeLattice(2, 4)
system_target = HeisenbergJ1J2(
    lattice=lat_target,
    J2=J2,
    use_symmetries=True,
    spin_inversion=1,
    ground_state_cache_dir=Path("groundstates"),
    skip_symmetries_whitelist=True,
)

system_source.get_eigenstates(1)
system_target.get_eigenstates(1)
target = 'amplitude'
X, Y = get_X_Y(system_source, sample=None if system_source.canonical_basis.states.shape[0] < 50000 else 50000)
X_target, Y_target = get_X_Y(system_target, sample=10000, target=target)

2023-06-20 13:31:35.467 | WARNING  | heisenberg_hamiltonians:__init__:414 - Symmetries are not tested with KagomeLattice2x3, and can produce incorrect results. Using them anyway due to skip_symmetries_whitelist=True.
2023-06-20 13:31:35.468 | DEBUG    | heisenberg_hamiltonians:__init__:446 - number_spins=18
2023-06-20 13:31:35.473 | DEBUG    | heisenberg_hamiltonians:__init__:456 - Symmetry group contains 12 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-06-20 13:31:35.528 | DEBUG    | heisenberg_hamiltonians:__init__:465 - Hilbert space dimension is 2102
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-06-20 13:31:35.573 | DEBUG    | heisenberg_hamiltonians:__init__:446 - number_spins=24
2023-06-20 13:31:35.576 | DEBUG    | heisenberg_hamiltonians:__init__:456 - Symmetry group contains 16 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-06-20 13:31:35.668 | DEBUG    | heisenberg_hamiltonians:__init__:465 - Hilbert spa

In [27]:
# Initialize the model
model = CNNRegression(kernel_size=2)

# Check if a GPU is available and if not, use a CPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Move the model to the device (GPU or CPU)
model.to(device)

# Create a TensorDataset from your inputs X and Y
dataset = TensorDataset(X, Y)

# Create a DataLoader for your dataset with a batch size of 32 and shuffling
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Define a loss function - Mean Squared Error (MSE) for regression
criterion = torch.nn.MSELoss()

# Define an optimizer - Adam
optimizer = optim.Adam(model.parameters(), lr=1e-3)  # Learning rate

# Number of epochs (iterations over the entire dataset)
epochs = 500

for epoch in range(epochs):
    running_loss = 0.0
    for i, data in enumerate(dataloader, 0):
        # Get the inputs and move them to the specified device
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)

        # Compute loss
        loss = criterion(outputs, labels.view(-1, 1))  # Reshape labels to match output shape

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Print statistics
        running_loss += loss.item()
        
        # print(f"{loss.item()=}")
        # print(f"Overlap: {overlap(model(X).view(-1), Y)}")
    # Print average loss per epoch
    print(f"Epoch {epoch + 1} loss: {running_loss / len(dataloader)}")
    print(f"Overlap: {overlap(torch.exp(model(X).view(-1)), torch.exp(Y))}")
    print(f"Overlap target: {overlap(torch.exp(model(X_target).view(-1)), torch.exp(Y_target))}")
    

print("Finished Training")


Epoch 1 loss: 3.1181071937084197
Overlap: 0.7660436630249023
Overlap target: 0.6009336709976196
Epoch 2 loss: 1.477696821681763
Overlap: 0.8095394968986511
Overlap target: 0.6210336685180664
Epoch 3 loss: 1.3497014016305144
Overlap: 0.8541601300239563
Overlap target: 0.6571345925331116
Epoch 4 loss: 1.2596099780578363
Overlap: 0.8828697204589844
Overlap target: 0.6953960061073303
Epoch 5 loss: 1.167973940799895
Overlap: 0.9067709445953369
Overlap target: 0.7122937440872192
Epoch 6 loss: 1.1036976439388175
Overlap: 0.9040916562080383
Overlap target: 0.7225042581558228
Epoch 7 loss: 1.0474922336834043
Overlap: 0.900556206703186
Overlap target: 0.7216516137123108
Epoch 8 loss: 0.9998610124776238
Overlap: 0.8996617197990417
Overlap target: 0.7142947316169739
Epoch 9 loss: 0.9744834065829453
Overlap: 0.9094261527061462
Overlap target: 0.7192517518997192
Epoch 10 loss: 0.9432784086485443
Overlap: 0.9001583456993103
Overlap target: 0.7224046587944031
Epoch 11 loss: 0.9124608741583009
Overlap:

KeyboardInterrupt: 